# FAβ-Gal - Fluorescence Analysis of senescence-associated β-Gal 

This Jupyter Notebook is designed to guide the user through the analysis of fluorescence images of cell culture or tissue sections senescence-associated beta-galactosidase (SA-βgal) assays, enabling a quantitative approach to this routine technique in aging studies.

The FAβ-Gal workflow first loads user images located in the specified **input_folder**. Input images must be in TIFF format (.tif or .tiff), and be taken using the same objective, laser intensity and time exposure for the far-red channel (βgal channel). 


## Options

The FAβ-Gal pipeline has two optional actions:

- **delete_intermediate_files**:\
When set to `True` (default), removes intermediate files generated by BiaPy at the end of the pipeline execution. These files can be useful if you are adjusting BiaPy parameters.

- **substract_background**:\
When set to `True` (default), performs the substract background algorithm nuceli channel of input images. This algorithm reduces the background fluorescence enhacing Biapy nuclei segmentation so we strongly recommend it. The radius parameter of the algorithm can be modified, because it should be bigger than the radius of the smallest object that you want to segmentate, hence, bigger than the radius of the nuclei. For more details you can check the code function in **function.py**

In [1]:
delete_intermediate_files = True
substract_background = False 

## Parameters

FAβ-Gal pipeline has the following parameters:

- **input_folder**:\
Path to input images directory (Must exist).

- **experiment_name**:\
Experiment name for naming output files generated in the analysis.

- **img_to_ind**:\
Path to tab-separated file with 2 columns: File (filenames of images) and Individual (biological replicate, eg. well, to which the image belongs). This file is used to compute the CTF per nuceli (for culture cell analyses) not only for each image but also at the biological replicate level (e.g. well), putting together the data of all the technical replicates (e.g. images of the same well). We strongly recommend this because it mitigates the effect of possible outliers such as images with a low number of cells. When set to `None` (default) FAβ-Gal only calculates the CTF per nuclei for aech image.


<div align="center">

**Example `img_to_ind.tsv`**

| File           | Individual |
|----------------|------------|
| image_001.tif  | A01        |
| image_002.tif  | A01        |
| image_003.tif  | A02        |
| image_004.tif  | B01        |
| image_005.tif  | B01        |

</div>

- **nuclei_ch**:\
Channel containing nuclei stain signal. Channels follow a 0-based numbering (first channel is 0). If only β-Gal stain signal is present in the image (e.g. tissue images with no nuclei staining), set as `None`.

- **bgal_ch**:\
Channel containing β-Gal stain signal (far-red).

- **pixel_size**:\
If pixel size is to be entered manually, change it here. Defaults to `None`, where the software will get the info from the image or, if no info is found, use 1 as default

- **backgr_val**:\
 Mean intensity of β-Gal channel in a background image (no cells and no debri) obtained elsewhere to be entered manually (e.g. ImageJ). If `None`, the software will calculate it using `backgr_img`.
 
 - **backgr_img**:\
Name of an image ( with no cells and no debri) store in `input_folder` from which the background mean intensity will be calculated if no `backgr_val` is entered. If both `backgr_val` and `backgr_img` are `None`, neither CTF/nuclei nor CTF/area will be calculated (see **results** section).

 - **bgal_th**:\
Minimum value of BGal stain signal to be considered positive. Default: 45. It should be set using a negative control (unstained image, inspected using e.g. ImageJ) and it corrects for cell autofluorescence. It is also advisable to use a positive control to avoid setting a threshold too restrictive.

 - **sbg_rad**:\
Radius of substract background algorithm (important only if `substract_background = True`, see **Parameters** above). Should be adjusted depending on nuclei size. Default: 31.

 - **config_file**:\
Path to BiaPy YAML configuration file, defaults to  `instance_segmentation.yaml`, it can be modified if necessary (for further information check [BiaPy documentation](https://biapy.readthedocs.io/en/latest/)).

 - **biapy_result_dir**:\
Path to directory to store BiaPy results, deafult is `./biapy_output`

 - **run_id**:\
Completar APU

 - **gpu**:\
GPU to use (as string, e.g., "0"). Obtained from `nvidia-smi` command.


In [2]:
# Files
input_folder="individuals" 
experiment_name = "Experiment1"
img_to_ind = "img_ind.tsv"
results_dir = "Results"

# Image parameters
nuclei_ch = None 
bgal_ch = 0 
pixel_size = None 

# β-Gal background parameters
backgr_val = 15 
backgr_img = None 

# β-Gal parameters
bgal_th = 90

# Substract background parameters (for nuclei)
sbg_rad = 31

# Biapy parameters
config_file = "instance_segmentation.yaml"
biapy_result_dir = "biapy_output"               
job_name = experiment_name                      # Name of BiaPy job. Dafaults to experiment name.
run_id = 1                                  
gpu = "0"                                   

## Measure β-Gal and prepare BiaPy input

Once you have stablished the options and the parameters run the following chunk to obtain the results of β-Gal quantification.


In [ ]:
from bioio import BioImage
from bioio_base.exceptions import UnsupportedFileFormatError
from pathlib import Path
from re import sub
from bioio_ome_tiff.writers import OmeTiffWriter
from functions import calculate_bgal, subtract_background  # Local file with B-Gal quantification and substract background functions.

# Create BiaPy runtime yml file

# Read all files
try:
    input_folder = Path(input_folder)
   
    if not input_folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {input_folder}")

    if not input_folder.is_dir():
        raise NotADirectoryError(f"Not a directory: {input_folder}")

except Exception as e:
    print(f"ERROR: Folder validation failed: {e}")

allowed_ext = {".tif", ".tiff"}
myfiles = list(input_folder.iterdir())

for inf in myfiles:
    #Exclude hidden files from filetype check
    if inf.name.startswith("."):
        continue
    if inf.is_file() and inf.suffix.lower() not in allowed_ext:
        raise ValueError(f"Invalid file found: {inf.name}. Please ensure all items in input folder are .tif or .tiff.")
    
# Iterate over all files
bres = {}
for inf in myfiles:
    if inf.name.startswith("."):
        continue

    print("Doing file " + inf.name)

    try:
        img = BioImage(inf)
    except UnsupportedFileFormatError as e:
        raise UnsupportedFileFormatError("\nERROR: Image is not in TIFF format, convert it to TIFF using appropiate software (e.g. ImageJ)") from e
    except Exception as e:
        raise Exception(f"\nERROR: Could not open {inf.name}: {e}") from e
        
    # Measure bgal area    
    try:
        bres[sub(r'(?i)\.tiff?$', '', inf.name)] = calculate_bgal(img,bgal_ch,bgal_th,psize=pixel_size)
    except IndexError as e:
        raise IndexError(f"\nERROR: Image {inf.name} does not have B-Gal input channel. Please check that B-Gal channel parameters are correct.") from e


    # Save tiff for biapy input (if nuclei channel is present)
    if nuclei_ch is not None:
        biapy_input = Path("biapy_input")
        biapy_input.mkdir(exist_ok=True)
        out_path = biapy_input / inf.name
        
        try:
            imgdata = img.get_image_data("YX",C=nuclei_ch)
        except IndexError as e:
            raise IndexError(f"\nERROR: Image does not have nuclei input channel. Please check that nuclei channel parameters are correct.") from e
        
        if substract_background:
            OmeTiffWriter.save(
                subtract_background(imgdata,
                                    radius=sbg_rad), 
                        str(out_path),
                        dim_order="YX")
        else:
            OmeTiffWriter.save(imgdata,
                        str(out_path),
                        dim_order="YX")

# Save bgal results
results_dir = Path(results_dir)
results_dir.mkdir(exist_ok=True)

BGalres = results_dir / f"{experiment_name}_Raw_BGal_results.tsv"
with BGalres.open('w') as bgal_f:
    bgal_f.write("File\tNpxPos\tNpxTot\tAreaPos\tAreaTot\tBgal_RawIntDen\n")
    for k, (npx, npxtot, areapos, areatot, RawIntDen) in bres.items():
        bgal_f.write(f"{k}\t{npx}\t{npxtot}\t{areapos}\t{areatot}\t{RawIntDen}\n")


Attempted file (/Users/droiva/Documents/FAB-Gal/Python/individuals/LAKI_390.ome.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/Users/droiva/Documents/FAB-Gal/Python/individuals/LAKI_390.ome.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6
Attempted file (/Users/droiva/Documents/FAB-Gal/Python/individuals/LAKI_390.ome.tif) load with reader: <class 'bioio_ome_tiff.reader.Reader'> failed with error: bioio-ome-tiff does not support the image: '/Users/droiva/Documents/FAB-Gal/Python/individuals/LAKI_390.ome.tif'. Failed to parse XML for the provided file. Error: not well-formed (invalid token): line 1, column 6
The image ends with .ome.tiff, which might indicate an OME-TIFF file format. You might want to install the `bioio-ome-tiff` plug-in for improved metadata Processing.You can also use 'bioio.plugin_feasibility_report(image)' method to check if a specif

Doing file LAKI_390.ome.tif
Image pixel sizes are PhysicalPixelSizes(Z=None, Y=0.5476193344672704, X=0.5476193344672704)
Doing file WT_399_2.ome.tif
Image pixel sizes are PhysicalPixelSizes(Z=None, Y=0.5476193344672704, X=0.5476193344672704)


## Run Biapy

Run BiaPy to quantify the number of nuclei if using FAβ-Gal for culture cells, if using it for tissue sections skip this chunk of code.

In [ ]:
# Create and run the BiaPy job

from biapy import BiaPy
biapy = BiaPy(config_file, result_dir=biapy_result_dir, name=job_name, run_id=run_id, gpu=gpu)
biapy.run_job()

## Process BiaPy's output

After running Biapy, adjust nuclei masks generated to facilitate segmentation check and condense BiaPy results of all images in a single file.

In [ ]:
import pandas as pd
from skimage import io
from skimage.color import label2rgb
import numpy as np
import shutil

# If BiaPy ended correctly and substract_background was not active, we first delete the biapy_input folders
if not substract_background:
    shutil.rmtree(biapy_input)

# BiaPy output folder
biapy_result_dir = Path(biapy_result_dir)
biapyout = biapy_result_dir / job_name / "results" / f"{job_name}_{run_id}" / "per_image_instances"

# Nuclei stats output file
biapystatsfile = results_dir / f"{experiment_name}_BiaPy_results.tsv"

### Process output mask images

Reads all mask images, and convert them to RGB, so they can be easily viasualize to be how segmentation has been done.

In [ ]:
for inf in biapyout.glob(".tif"):
    img = io.imread(inf)
    img = label2rgb(img)*255
    img = img.astype(np.uint8)
    io.imsave(biapyout / inf.stem / ".png",img)
    inf.unlink()

### Process nuclei stats

Reads all CSVs generated by BiaPy, adds file name and concatenates them in a single file

In [ ]:
#list of csv with file name column
biapy_csv = list(biapyout.glob("*full_stats.csv"))

dfs = []
for inf in biapy_csv:
    df = pd.read_csv(inf)
    df["File"] = inf.stem.replace("_full_stats","")
    dfs.append(df)

#concatenate
result = pd.concat(dfs, ignore_index=True)
result = result.drop(columns=["conditions"])
result.to_csv(str(biapystatsfile), sep="\t")

# Delete original csv
for inf in biapy_csv:
    inf.unlink()

# Merge nuclei and β-Gal stats

In [ ]:
# Load BGal and nuclei stats

nucleidf = pd.read_table(biapystatsfile)
bgaldf = pd.read_table(results_dir / f"{experiment_name}_Raw_BGal_results.tsv")

# Count nuclei per image file 
nucleitot = nucleidf['File'].value_counts()

# B-Gal background intensity
computeCTF = True
if backgr_val is not None:
    BGal_backgr = backgr_val
elif backgr_img is not None:
    BGal_backgr = bgaldf.at[backgr_img, 'BGal_RawIntDen'] / bgaldf.at[backgr_img, 'NpxTot']
else:
    computeCTF = False

# Merge nuclei and B-Gal data
resdf = pd.merge(bgaldf,nucleitot,how='outer',on='File')
resdf = resdf.rename(columns = {'count':'NumNucl'})

# Calculate CTF on individual images (only if background intensity is present)
if computeCTF:
    CTFimg = resdf
    CTFimg['bgMF'] = BGal_backgr
    CTFimg['CTFnucl'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NumNucl
    CTFimg['CTFpix'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NpxTot
    CTFimg['CTFarea'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.AreaTot

    CTFimg.to_csv(results_dir / f"{experiment_name}_results_perimage.tsv", sep="\t")

    # Load individual info (if present)
    if img_to_ind is not None:
        img_ind_df = pd.read_table(img_to_ind)
        img_ind_df["File"] = img_ind_df["File"].str.replace(r"(?i)\.tiff?$","",regex=True)
        resdf = pd.merge(resdf,img_ind_df,how='outer',on='File')

        CTFind = resdf.groupby(['Individual']).sum(numeric_only = True)
        
        CTFind['bgMF'] = BGal_backgr
        CTFind['CTFnucl'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NumNucl
        CTFind['CTFpix'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NpxTot
        CTFind['CTFarea'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.AreaTot

        CTFind.to_csv(results_dir / f"{experiment_name}_results_perindividual.tsv")

### Delete extra files generated by Biapy (optional)

Delete intermediate files and files not used in this analysis

In [ ]:
if delete_intermediate_files:
    interf = biapy_result_dir / job_name / "results" / f"{job_name}_{run_id}" / "per_image"
    extraf = biapy_result_dir / job_name / "results" / f"{job_name}_{run_id}" / "per_image_post_processing"

    shutil.rmtree(interf)
    shutil.rmtree(extraf)